In [ ]:
import arcpy
import os

aprx = arcpy.mp.ArcGISProject("CURRENT")
map_obj = aprx.listMaps("CompHSI_map")[0]
layers = map_obj.listLayers()
tpkx_folder = r"C:\Users\kyashby\Documents\PROJECTS\Nooksack\Substrate_Cover_MappingSupportNooksack\TPKX_FILES"
lyrx_folder = r"C:\Users\kyashby\Documents\PROJECTS\Nooksack\Substrate_Cover_MappingSupportNooksack\LYRX_files\Comp_HSI"

# Before running this code, make sure to save the lyrx files of the rasters WITH the layers turned on and with the needed symbology applied
# MAKE SURE THE BASEMAP IS ALSO REMOVED FROM THE SPECIFIED MAP

# The code below will use the lyrx files to add the rasters one at a time to the specified map and create a map package for that layer.
# If the code crashes partway through, be sure to delete the last package it was working and restart the code.
# The code will skip the layers that already have tpkx files created.
lyrx_count = 0
lyrx_total = len(os.listdir(lyrx_folder))
for lyrx in os.listdir(lyrx_folder):
    if lyrx.endswith(".lyrx"):
        print(f"Adding {lyrx}")
        lyrx_path = os.path.join(lyrx_folder, lyrx)
        lyrFile = arcpy.mp.LayerFile(lyrx_path)
        map_obj.addLayer(lyrFile)
        layers = map_obj.listLayers("*")
        tpkx_files_list = os.listdir(tpkx_folder)
        for layer in layers:
            if "OMY" in layer.name:
                fish_tag = "Steelhead"
            elif "CHK" in layer.name:
                fish_tag = "Chinook Salmon"
            elif "CHM" in layer.name:
                fish_tag = "Chum Salmon"
            elif "COH" in layer.name:
                fish_tag = "Coho Salmon"
            elif "BTDV" in layer.name:
                fish_tag = "Bull Trout/Dolly Varden"
            elif "CTT" in layer.name:
                fish_tag = "Cutthroat Trout"
            if f"{layer}.tpkx" in tpkx_files_list:
                print(f"{layer}.tpkx file already exists")
                map_obj.removeLayer(layer)
            else:
                print(f"Creating {layer.name}.tpkx with {fish_tag} in tags")
                arcpy.management.CreateMapTilePackage(
                    in_map=map_obj,
                    service_type="ONLINE",
                    output_file=os.path.join(tpkx_folder, f"{layer.name}.tpkx"),
                    min_level_of_detail=15,
                    level_of_detail=20,
                    summary=f"This is the tiling map package for {layer.name}",
                    tags=f"{fish_tag}",
                )
                lyrx_count += 1
                print(f"Created {layer.name}.tpkx, {lyrx_count}/{lyrx_total}")
                map_obj.removeLayer(layer)



In [ ]:
import arcpy
import os
tpkx_folder = r"C:\Users\kyashby\Documents\PROJECTS\Nooksack\Substrate_Cover_MappingSupportNooksack\TPKX_FILES"
tpkx_file_list = os.listdir(tpkx_folder)
files = [f for f in tpkx_file_list if os.path.isfile(os.path.join(tpkx_folder, f))]
for file in files:
    if file.endswith(".tpkx"):
        if "omy" in file.lower():
            fish_tag = "Steelhead"
        elif "chk" in file.lower():
            fish_tag = "Chinook Salmon"
        elif "chm" in file.lower():
            fish_tag = "Chum Salmon"
        elif "coh" in file.lower():
            fish_tag = "Coho Salmon"
        elif "btdv" in file.lower():
            fish_tag = "Bull Trout/Dolly Varden"
        elif "ctt" in file.lower():
            fish_tag = "Cutthroat Trout"
        try:
            print(f"Publishing {file}")
            result = arcpy.management.SharePackage(
                in_package=f"{tpkx_folder}\\{file}",
                summary=f"Raster results for {os.path.splitext(file)[0]}",
                tags=fish_tag,
                public="MYGROUPS",
                organization="MYORGANIZATION",
                groups=None,
                publish_web_layer="TRUE",
                portal_folder="Nooksack_Whatcom_HSI_rasters"
            )
            messages = result.getMessages()
            if "succeeded" in messages.lower():
                print(f"{file} published successfully")
                print(messages)
        except Exception as e:
            print(f"Error publishing {file}")
            print(e)

In [2]:
import os
import arcpy
aprx = arcpy.mp.ArcGISProject("CURRENT")

def publish_tile_package(tpkx_folder,tag,portal_folder):
    tpkx_file_list = os.listdir(tpkx_folder)
    files = [f for f in tpkx_file_list if os.path.isfile(os.path.join(tpkx_folder, f))]
    for file in files:
        if file.endswith(".tpkx"):
            try:
                print(f"Publishing {file}")
                result = arcpy.management.SharePackage(
                    in_package=f"{tpkx_folder}\\{file}",
                    summary=f"Raster results for {os.path.splitext(file)[0]}",
                    tags=tag,
                    public="MYGROUPS",
                    organization="MYORGANIZATION",
                    groups=None,
                    publish_web_layer="TRUE",
                    portal_folder=portal_folder
                )
                messages = result.getMessages()
                if "succeeded" in messages.lower():
                    print(f"{file} published successfully")
                    print(messages)
            except Exception as e:
                print(f"Error publishing {file}")
                print(e)

def create_tile_packages(map_name, lyrx_folder, tpkx_folder, tag):
    map_obj = aprx.listMaps(f"{map_name}")[0]
    lyrx_count = 0
    lyrx_total = len(os.listdir(lyrx_folder))
    for lyrx in os.listdir(lyrx_folder):
        if lyrx.endswith(".lyrx"):
            print(f"Adding {lyrx}")
            lyrx_path = os.path.join(lyrx_folder, lyrx)
            lyrFile = arcpy.mp.LayerFile(lyrx_path)
            map_obj.addLayer(lyrFile)
            layers = map_obj.listLayers("*")
            tpkx_files_list = os.listdir(tpkx_folder)
            for layer in layers:
                if f"{layer}.tpkx" in tpkx_files_list:
                    print(f"{layer}.tpkx file already exists")
                    map_obj.removeLayer(layer)
                else:
                    print(f"Creating {layer.name}.tpkx")
                    arcpy.management.CreateMapTilePackage(
                        in_map=map_obj,
                        service_type="ONLINE",
                        output_file=os.path.join(tpkx_folder, f"{layer.name}.tpkx"),
                        min_level_of_detail=15,
                        level_of_detail=20,
                        summary=f"This is the tiling map package for {layer.name}",
                        tags=f"{tag}",
                    )
                    lyrx_count += 1
                    print(f"Created {layer.name}.tpkx, {lyrx_count}/{lyrx_total}")
                    map_obj.removeLayer(layer)
publish_tile_package(
    tpkx_folder=r"C:\Users\kyashby\Documents\PROJECTS\Nooksack\Substrate_Cover_MappingSupportNooksack\TPKX_FILES\Depth_tiles",
    tag="Depth raster",
    portal_folder="Nooksack_Whatcom_HSI_rasters"
)



IndentationError: expected an indented block after 'try' statement on line 11 (627581303.py, line 12)